[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/slm-development/blob/main/tutorial/01_small_language_models_what_they_are_and_why_they_matter/02_tokens_probabilities_prediction.ipynb)

# How a Language Model Learns: Tokens, Probabilities, and Prediction

| | |
|---|---|
| **Domain** | GenAI |
| **Module** | Small Language Models: What They Are and Why They Matter |
| **Difficulty** | Beginner |
| **Estimated Time** | 30 minutes |
| **Prerequisites** | Basic Python programming knowledge; familiarity with what a model is and what training vs. inference means; no prior deep learning or NLP experience required |

---

## Lesson Roadmap

- **Core Concepts** — Understand tokenization, next-token prediction, and probability distributions using concrete analogies
- **Technical Deep-Dive** — Run a working tokenizer and observe softmax outputs in Python
- **Hands-On Exercise** — Tokenize a custom string, decode it back, and inspect a raw probability distribution
- **Concept Check** — Four questions testing tokenization, BPE, next-token prediction, and softmax

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain tokenization and describe what a token is without using jargon
- Describe the next-token prediction objective and explain why it drives language model training
- Interpret a simple probability distribution over a vocabulary as a model output
- Recognize how byte-pair encoding (BPE) compresses vocabulary, citing Sennrich et al. (2016)

---

## ⚙️ Setup — Run This Cell First

In [ ]:
# ── Auto-install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.41.2 torch==2.3.0 accelerate==0.30.1 sentencepiece==0.2.0

import transformers, torch
print(f"✅ transformers {transformers.__version__}")
print(f"✅ torch {torch.__version__}")
print("✅ Setup complete!")

---

## 🟢 Core Concepts

### What Is a Token?

A language model never reads characters or words directly. It reads **tokens** — small chunks of text that sit between individual characters and full words in size.

Consider this sentence:

```
"Rainfall predictions help farmers plan irrigation."
```

A tokenizer might split it into:

```
["Rain", "fall", " predictions", " help", " farmers", " plan", " irrigation", "."]
```

Notice that `"Rainfall"` splits into two tokens. A less common word gets broken into recognizable sub-pieces. A very common word like `"plan"` stays whole. This follows a learned compression strategy called **byte-pair encoding**.

### Byte-Pair Encoding (BPE)

Sennrich et al. (2016) introduced BPE as a way to build a compact, fixed-size vocabulary that handles rare words gracefully. The algorithm starts with individual characters and repeatedly merges the most frequent adjacent pair into a new token.

> **Note:** BPE is the tokenization strategy behind GPT-2, GPT-4, and many Hugging Face models including SmolLM2. The vocabulary size for SmolLM2-135M is 49,152 tokens.

### Next-Token Prediction: The Core Training Signal

Every autoregressive language model trains by solving one repeated task:

> **Given the tokens seen so far, predict the next one.**

The model produces a **probability distribution** across all 49,152 vocabulary entries. The correct token should receive the highest score.

### Softmax and Temperature

The **softmax** function converts raw scores (logits) into a valid probability distribution that sums to 1.0.

**Temperature** controls how sharp or flat this distribution becomes:

- **Temperature < 1.0** (e.g., 0.3): Distribution sharpens — the top token dominates
- **Temperature = 1.0**: Use the raw softmax distribution
- **Temperature > 1.0** (e.g., 1.8): Distribution flattens — more exploratory

---

## 🔷 Technical Deep-Dive

### Step 1: Tokenize a String with SmolLM2's Tokenizer

Let's see BPE in action. The model downloads automatically (~270 MB first time).

In [ ]:
"""
Demonstrates BPE tokenization using SmolLM2-135M's tokenizer.
SmolLM2 uses a 49,152-token BPE vocabulary.
"""

from transformers import AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

sample_text = "Rainfall predictions help farmers plan irrigation schedules efficiently."

# Encode to token IDs
token_ids = tokenizer.encode(sample_text)

# Map each ID back to its string representation
token_strings = [tokenizer.decode([tid]) for tid in token_ids]

print(f"Original text : {sample_text}")
print(f"Token IDs     : {token_ids}")
print(f"Token strings : {token_strings}")
print(f"Token count   : {len(token_ids)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

> **Observe:** `"Rainfall"` splits into `"Rain"` + `"fall"` because the merged token `"Rainfall"` sits below BPE's frequency threshold. This is BPE in action.

### Step 2: Build a Softmax Distribution from Raw Logits

This block requires **no model download**. It demonstrates softmax and temperature using a hand-crafted logit vector — the same computation a model performs on its output layer.

In [ ]:
"""
Simulates the final output layer of a language model over a toy vocabulary.
Demonstrates how temperature reshapes the probability distribution.
"""

import torch
import torch.nn.functional as F

# Toy vocabulary — five tokens from a domain-specific sensor dataset
VOCAB = ["anomaly", "normal", "error", "timeout", "offline"]

# Raw logit scores (as a model might output before softmax)
raw_logits = torch.tensor([4.8, 2.1, 1.3, 0.7, -0.5])

def show_distribution(logits, temperature, vocab):
    """Print softmax probabilities for a given temperature."""
    scaled_logits = logits / temperature
    probabilities = F.softmax(scaled_logits, dim=0)

    print(f"\n  Temperature = {temperature}")
    print(f"  {'Token':<12} {'Probability':>12}")
    print(f"  {'-'*26}")
    for token, prob in zip(vocab, probabilities):
        bar = "█" * int(prob.item() * 30)
        print(f"  {token:<12} {prob.item():>11.4f}  {bar}")

print("Logit vector:", dict(zip(VOCAB, raw_logits.tolist())))

for temp in [0.3, 1.0, 1.8]:
    show_distribution(raw_logits, temperature=temp, vocab=VOCAB)

**What you should observe:**
- At temperature `0.3`, the model almost always picks `"anomaly"` (~99.9%)
- At temperature `1.0`, `"anomaly"` still dominates (~82%) but other tokens have meaningful probability
- At temperature `1.8`, the distribution spreads — `"anomaly"` is only ~55%

### Step 3: Decode Tokens Back to Text

Confirms that tokenization is lossless: encode → decode returns the original string.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

original = "Sensor array calibration failed at sector 7-G."

token_ids = tokenizer.encode(original, add_special_tokens=False)
reconstructed = tokenizer.decode(token_ids, skip_special_tokens=True)

assert reconstructed == original, (
    f"Decode mismatch!\nOriginal    : {original}\nReconstructed: {reconstructed}"
)

print(f"✅ Lossless round-trip confirmed.")
print(f"   Tokens used: {len(token_ids)}")
print(f"   Token IDs  : {token_ids}")

---

## 🧪 Hands-On Exercise

**Goal:** Tokenize three different sentence types and compare how BPE handles common vs. rare vocabulary.

**Time:** ~10 minutes

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

sentences = [
    "The model trained on structured tabular data.",          # common vocabulary
    "Phytoremediation techniques detoxify contaminated soil.", # rare/scientific
    "42 / 0 = ??? 🤖",                                        # mixed: numbers, symbols, emoji
]

for sentence in sentences:
    token_ids = tokenizer.encode(sentence)
    token_strings = [tokenizer.decode([tid]) for tid in token_ids]
    single_char_tokens = [t for t in token_strings if len(t.strip()) == 1]

    print(f"\nText:   {sentence}")
    print(f"Tokens: {token_strings}")
    print(f"Count:  {len(token_ids)}")
    print(f"Single-char tokens: {single_char_tokens if single_char_tokens else 'None'}")

print("\n" + "="*60)
print("✅ Exercise complete!")
print("Observe: 'Phytoremediation' splits into more tokens than 'trained'.")
print("The emoji 🤖 produces multiple token IDs (UTF-8 byte tokens).")

---

## ❓ Concept Check

### Q1: A tokenizer splits `"electroencephalography"` into eight sub-word pieces. What does this tell you?

- **A)** The word appeared rarely, so BPE never merged its components into a single token.
- **B)** The word appeared frequently, which caused BPE to split it for efficiency.
- **C)** The tokenizer made an error.
- **D)** Sub-word count is determined by word length, not frequency.

<details><summary>🔑 Answer</summary>

**A.** BPE merges frequent adjacent pairs. A rare word never triggers enough merges to become a single token.
</details>

### Q2: A model outputs logits `[3.1, 0.2, -1.4]`. After softmax, which token gets highest probability?

- **A)** The token with logit 3.1 — softmax is order-preserving.
- **B)** The token with logit 0.2.
- **C)** The token with logit -1.4.
- **D)** Softmax randomizes the order.

<details><summary>🔑 Answer</summary>

**A.** Softmax preserves rank order. The highest logit always maps to the highest probability.
</details>

### Q3: You set temperature to 0.1. What effect?

- **A)** Distribution becomes flatter.
- **B)** No effect below 1.0.
- **C)** Distribution sharpens — the top token dominates strongly.
- **D)** Switches to greedy decoding.

<details><summary>🔑 Answer</summary>

**C.** Dividing logits by a small temperature amplifies differences, creating a peaked distribution.
</details>

---

## Summary

- **Tokens are sub-word units**, not characters or words. BPE (Sennrich et al., 2016) builds the vocabulary by merging frequent adjacent character pairs.
- **Next-token prediction is the training objective.** The model estimates a probability distribution over its full vocabulary and adjusts weights to increase the probability of the correct next token.
- **Softmax converts logits to probabilities.** Temperature scales the logits before softmax: low temperature sharpens; high temperature spreads.
- **Attention lets each token consider all others.** The transformer’s attention mechanism (Vaswani et al., 2017) computes weighted relationships between every pair of tokens.

---

## References & Credits

- Sennrich et al. (2016) *Neural Machine Translation of Rare Words with Subword Units.* [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
- Vaswani et al. (2017) *Attention Is All You Need.* [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- HuggingFaceTB. *SmolLM2-135M Model Card.* [HF Hub](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) — Last verified: 2025-06